# Phase 6 — Evaluation on the Test Period

**Purpose.** Score the nine models of Phase 5 on the Test period **once**, with frozen settings, and answer three questions: how good are the scores out of time, does the mortgage rate add anything on Test (where the 2022-23 rate-hiking cycle sits), and how should the predictions be used (decision rules, calibration, stability over time).

**Nothing in this notebook is allowed to change a model, a setting or a decision rule after Test has been seen.** Every choice was made in Phase 5 on the validation folds, and the one cut-off chosen here (the F1 threshold) is chosen on the saved validation predictions and written to a file **before** Test is scored.

## Design, fixed before any result

| Item | Choice |
|---|---|
| Models | All 9 of Phase 5 (3 families x 3 feature steps), same fixed settings, refit once on **all** Train rows |
| Test scoring | Once, with frozen models. Results are locked in `outputs/phase6_test_lock.json`. A re-run must reproduce them or the notebook stops |
| Metrics | ROC-AUC (main), PR-AUC and Brier score, read against the Test base rate (8.11%) |
| Does the rate help on Test? | Same rule as Phase 5: the interval of the change excludes zero **and** the gain is at least +0.01 ROC-AUC. A step is robust if it helps in at least 2 of the 3 families. Bootstrap over the 17 Test quarters, 2,000 draws |
| Decision rules | Flag the riskiest 5%, 10% and 20% of counties **in each quarter**, and flag every county whose probability is at or above the F1-maximizing threshold found on the validation predictions. No rule is picked: all four are reported side by side |
| Reference | The same rules computed on the saved validation predictions, so Test can be compared with validation |
| Calibration | Reported on Test and on validation, with no recalibration |
| Stability | ROC-AUC, PR-AUC and recall at the top 10% for each of the 17 Test quarters |

**What this notebook does not do.** It does not tune anything, does not pick a different model because of Test, and does not recalibrate.

## 1. Setup and connection

In [1]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             precision_recall_curve, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine, text

In [2]:
load_dotenv(override=True)

DB_USER = os.environ.get("PGUSER", "postgres")
DB_PASSWORD = os.environ.get("PGPASSWORD", "")
DB_HOST = os.environ.get("PGHOST", "localhost")
DB_PORT = os.environ.get("PGPORT", "5432")
DB_NAME = os.environ.get("PGDATABASE", "housing_risk_sql")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

os.makedirs("../outputs", exist_ok=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database();")).scalar())
    for name in ("quarterly_modeling_table", "phase5_validation_predictions", "phase5_step_summary"):
        assert conn.execute(text("SELECT to_regclass(:n)"), {"n": name}).scalar(), (
            f"STOP: {name} not found. Run notebooks 04 and 05 first.")
assert Path("../outputs/phase5_baseline_lock.json").exists(), "STOP: run notebook 05 first (baseline lock missing)."
print("quarterly_modeling_table, phase5_validation_predictions, phase5_step_summary and the baseline lock found.")

capstone_sql_variant
quarterly_modeling_table, phase5_validation_predictions, phase5_step_summary and the baseline lock found.


## 2. Load Train and Test, and check them

Train is every Train-split row of the modeling table (this includes 2021-Q1 to Q3, which sit in no validation fold). Test is the Test split. Embargo and unassigned rows are never read. The checks make sure Test starts after Train ends, and that nothing is missing.

In [3]:
FEATURES_ALL = ["momentum_1m", "momentum_3m", "momentum_6m",
                "rate_level", "rate_roc_12m", "rate_momentum_interaction"]
query = ("SELECT county_fips, quarter, label_quarter_end, split, decline_label, "
         + ", ".join(FEATURES_ALL) + " FROM quarterly_modeling_table WHERE split = '{}' "
         "ORDER BY county_fips, quarter")
train = pd.read_sql(query.format("train"), engine)
test = pd.read_sql(query.format("test"), engine)
for d in (train, test):
    for c in ("quarter", "label_quarter_end"):
        d[c] = pd.to_datetime(d[c])

test_quarters = sorted(test["quarter"].unique())
n_missing = int(train[FEATURES_ALL].isna().sum().sum() + test[FEATURES_ALL].isna().sum().sum())
print(f"Train: {len(train):,} rows, {train['county_fips'].nunique():,} counties, "
      f"{100 * train['decline_label'].mean():.2f}% declines, "
      f"labels from {train['label_quarter_end'].min().date()} to {train['label_quarter_end'].max().date()}")
print(f"Test:  {len(test):,} rows, {test['county_fips'].nunique():,} counties, "
      f"{100 * test['decline_label'].mean():.2f}% declines, {len(test_quarters)} quarters, "
      f"labels from {test['label_quarter_end'].min().date()} to {test['label_quarter_end'].max().date()}")
print(f"Missing feature values: {n_missing} | Train ends before Test starts: "
      f"{train['label_quarter_end'].max() < test['label_quarter_end'].min()}")
assert set(train["split"]) == {"train"} and set(test["split"]) == {"test"}
assert n_missing == 0, "A feature value is missing."
assert train["label_quarter_end"].max() < test["label_quarter_end"].min(), "Train reaches into Test."
assert len(test_quarters) == 17, "Expected 17 Test quarters."
if (len(train), len(test)) != (164460, 51980):
    print("NOTE: differs from the figures verified in notebook 04 (Train 164,460, Test 51,980).")

summary5 = pd.read_sql("SELECT * FROM phase5_step_summary", engine)
rec = summary5[summary5["is_recommended"].astype(bool)].iloc[0]
REC = (rec["family"], rec["step"])
print(f"Recommended in Phase 5 (validation only): {REC[0]} with {REC[1]} "
      f"(mean ROC-AUC {rec['mean_roc_auc']:.4f}, mean PR-AUC {rec['mean_pr_auc']:.4f})")

Train: 164,460 rows, 3,009 counties, 4.85% declines, labels from 2001-06-30 to 2021-09-30
Test:  51,980 rows, 3,071 counties, 8.11% declines, 17 quarters, labels from 2022-03-31 to 2026-03-31
Missing feature values: 0 | Train ends before Test starts: True
Recommended in Phase 5 (validation only): random_forest with baseline (mean ROC-AUC 0.8680, mean PR-AUC 0.3120)


## 3. Models and feature steps (frozen)

Exactly the families, settings and feature steps of Phase 5. Nothing here is changed after Test has been seen.

In [4]:
FEATURE_STEPS = {
    "baseline":          ["momentum_1m", "momentum_3m", "momentum_6m"],
    "step2_rate_level":  ["momentum_1m", "momentum_3m", "momentum_6m", "rate_level"],
    "step3_rate_change": FEATURES_ALL,
}
FAMILIES = ["logreg", "random_forest", "hist_gb"]
STEP_ORDER = list(FEATURE_STEPS)
MODELS = [(f, s) for f in FAMILIES for s in STEP_ORDER]
RANDOM_STATE = 42
N_BOOT = 2000
MIN_GAIN = 0.01
TOP_SHARES = [0.05, 0.10, 0.20]


def make_model(family):
    if family == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    if family == "random_forest":
        return RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_depth=12,
                                      random_state=RANDOM_STATE, n_jobs=-1)
    if family == "hist_gb":
        return HistGradientBoostingClassifier(max_iter=150, random_state=RANDOM_STATE)
    raise ValueError(family)

## 4. Decision rules chosen on validation, before Test is scored

Phase 5 saved the out-of-sample predictions of every model on the 11 validation folds. From them, **before Test predictions exist**, this section finds the probability cut-off that maximizes F1 (one cut-off per model, from all validation rows pooled) and computes the reference results of every decision rule.

The four rules are:

- **Top 5%, 10%, 20% in each quarter:** in every quarter, flag the counties with the highest predicted probability, as many as that share of the quarter's counties. The rule depends only on the ranking within a quarter, so it does not depend on the base rate.
- **F1 threshold:** flag every county whose probability is at or above the cut-off found here on validation, applied unchanged to Test. It does depend on the base rate, and Test has more declines than validation, so it will flag a different share there.

The cut-offs are written to `outputs/phase6_thresholds_lock.json` first. Test is scored only after that.

In [5]:
vp = pd.read_sql("SELECT * FROM phase5_validation_predictions ORDER BY fold_id, county_fips, quarter", engine)
vp["quarter"] = pd.to_datetime(vp["quarter"])
val_y = vp["decline_label"].to_numpy()
val_q = vp["quarter"].to_numpy()
val_p = {m: vp[f"proba_{m[0]}_{m[1]}"].to_numpy() for m in MODELS}
print(f"Validation predictions: {len(vp):,} rows, {int(val_y.sum()):,} declines, "
      f"{vp['quarter'].nunique()} quarters, {vp['fold_id'].nunique()} folds")
assert vp["quarter"].nunique() == 44 and vp["fold_id"].nunique() == 11, "Expected 44 validation quarters in 11 folds."
if (len(vp), int(val_y.sum())) != (112734, 4252):
    print("NOTE: differs from the figures verified in notebooks 04 and 05 (112,734 rows, 4,252 declines).")

Validation predictions: 112,734 rows, 4,252 declines, 44 quarters, 11 folds


In [6]:
def top_share_counts(y, p, q, share):
    # Flag the top `share` of rows inside each quarter; return TP, FP, FN, TN summed over quarters.
    tp = fp = fn = tn = 0
    for qv in np.unique(q):
        m = q == qv
        yq, pq = y[m], p[m]
        k = max(1, int(round(share * len(yq))))
        flag = np.zeros(len(yq), dtype=bool)
        flag[np.argsort(-pq, kind="mergesort")[:k]] = True
        tp += int((flag & (yq == 1)).sum()); fp += int((flag & (yq == 0)).sum())
        fn += int((~flag & (yq == 1)).sum()); tn += int((~flag & (yq == 0)).sum())
    return tp, fp, fn, tn


def threshold_counts(y, p, t):
    flag = p >= t
    return (int((flag & (y == 1)).sum()), int((flag & (y == 0)).sum()),
            int((~flag & (y == 1)).sum()), int((~flag & (y == 0)).sum()))


def rule_row(split, model, rule, counts):
    tp, fp, fn, tn = counts
    n = tp + fp + fn + tn
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec_ = tp / (tp + fn) if tp + fn else float("nan")
    base = (tp + fn) / n
    return {"split": split, "family": model[0], "step": model[1], "rule": rule,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn, "precision": prec, "recall": rec_,
            "flagged_share": (tp + fp) / n, "lift": prec / base if base else float("nan")}


RULE_NAMES = [f"top {int(100 * s)}% each quarter" for s in TOP_SHARES] + ["F1 threshold from validation"]

f1_threshold = {}
for m in MODELS:
    prec, rec_, thr = precision_recall_curve(val_y, val_p[m])
    f1 = 2 * prec * rec_ / (prec + rec_ + 1e-12)
    i = int(np.nanargmax(f1[:-1]))
    f1_threshold[m] = {"threshold": float(thr[i]), "val_precision": float(prec[i]),
                       "val_recall": float(rec_[i]), "val_f1": float(f1[i])}

rules_val = []
for m in MODELS:
    for s, name in zip(TOP_SHARES, RULE_NAMES[:3]):
        rules_val.append(rule_row("validation", m, name, top_share_counts(val_y, val_p[m], val_q, s)))
    rules_val.append(rule_row("validation", m, RULE_NAMES[3],
                              threshold_counts(val_y, val_p[m], f1_threshold[m]["threshold"])))
rules_val = pd.DataFrame(rules_val)

thr_table = pd.DataFrame([{"family": m[0], "step": m[1], **f1_threshold[m]} for m in MODELS])
print("F1-maximizing thresholds, found on validation predictions only:")
thr_table.round(4)

F1-maximizing thresholds, found on validation predictions only:


,family,step,threshold,val_precision,val_recall,val_f1
0,logreg,baseline,0.2805,0.4103,0.4929,0.4479
1,logreg,step2_rate_level,0.2573,0.4178,0.4847,0.4488
2,logreg,step3_rate_change,0.2379,0.3983,0.4960,0.4418
3,random_forest,baseline,0.3386,0.4039,0.4937,0.4443
4,random_forest,step2_rate_level,0.3327,0.3802,0.4650,0.4183
5,random_forest,step3_rate_change,0.3309,0.3956,0.4436,0.4182
6,hist_gb,baseline,0.3895,0.4359,0.4506,0.4432
7,hist_gb,step2_rate_level,0.2784,0.3285,0.5200,0.4027
8,hist_gb,step3_rate_change,0.2753,0.3314,0.4737,0.3900


In [7]:
THR_LOCK = Path("../outputs/phase6_thresholds_lock.json")
thr_lock = {"purpose": "F1 thresholds found on the Phase 5 validation predictions, written before Test was scored.",
            "validation_rows": int(len(vp)), "validation_declines": int(val_y.sum()),
            "thresholds": {f"{m[0]}|{m[1]}": f1_threshold[m] for m in MODELS}}
if THR_LOCK.exists():
    old = json.loads(THR_LOCK.read_text(encoding="utf-8"))
    gap = max(abs(old["thresholds"][k][f] - v[f]) for k, v in thr_lock["thresholds"].items() for f in v)
    assert gap < 1e-6, f"The thresholds no longer match the locked file. Delete {THR_LOCK} only if the validation predictions changed on purpose."
    print(f"Thresholds reproduce the locked file (largest gap {gap:.1e}).")
else:
    THR_LOCK.write_text(json.dumps(thr_lock, indent=2), encoding="utf-8")
    print(f"Thresholds locked in {THR_LOCK}")
rec_val = rules_val[(rules_val["family"] == REC[0]) & (rules_val["step"] == REC[1])]
print(f"\nValidation reference for the recommended model ({REC[0]}, {REC[1]}):")
rec_val.drop(columns=["split", "family", "step"]).round(4).reset_index(drop=True)

Thresholds locked in ..\outputs\phase6_thresholds_lock.json

Validation reference for the recommended model (random_forest, baseline):


,rule,tp,fp,fn,tn,precision,recall,flagged_share,lift
0,top 5% each quarter,1650,3983,2602,104499,0.2929,0.3881,0.0500,7.7662
1,top 10% each quarter,2351,8923,1901,99559,0.2085,0.5529,0.1000,5.5289
2,top 20% each quarter,3082,19467,1170,89015,0.1367,0.7248,0.2000,3.6238
3,F1 threshold from validation,2099,3098,2153,105384,0.4039,0.4937,0.0461,10.7083


## 5. Refit the nine models on all Train and score Test once

Each model is refit once on all Train rows with the Phase 5 settings, and predicts Test once. This is the only place where Test is scored. Nothing is selected from the result.

In [8]:
test_y = test["decline_label"].to_numpy()
test_p = {}
for family, step in MODELS:
    feats = FEATURE_STEPS[step]
    t0 = time.time()
    model = make_model(family).fit(train[feats].to_numpy(), train["decline_label"].to_numpy())
    test_p[(family, step)] = model.predict_proba(test[feats].to_numpy())[:, 1]
    assert not np.isnan(test_p[(family, step)]).any()
    print(f"{step:18s} {family:14s} fitted on {len(train):,} rows and scored on {len(test):,} in {time.time() - t0:5.1f}s")
assert len(test_p) == 9

baseline           logreg         fitted on 164,460 rows and scored on 51,980 in   0.2s
step2_rate_level   logreg         fitted on 164,460 rows and scored on 51,980 in   0.3s
step3_rate_change  logreg         fitted on 164,460 rows and scored on 51,980 in   0.4s
baseline           random_forest  fitted on 164,460 rows and scored on 51,980 in  24.7s
step2_rate_level   random_forest  fitted on 164,460 rows and scored on 51,980 in  32.4s
step3_rate_change  random_forest  fitted on 164,460 rows and scored on 51,980 in  26.9s
baseline           hist_gb        fitted on 164,460 rows and scored on 51,980 in   2.4s
step2_rate_level   hist_gb        fitted on 164,460 rows and scored on 51,980 in   2.5s
step3_rate_change  hist_gb        fitted on 164,460 rows and scored on 51,980 in   1.9s


## 6. Test scores

ROC-AUC, PR-AUC and Brier score for every model, with the Phase 5 validation average next to it. PR-AUC has to be read against the Test base rate, not against the validation one.

In [9]:
val_mean = summary5.set_index(["family", "step"])
rows = []
for m in MODELS:
    rows.append({"family": m[0], "step": m[1],
                 "test_roc_auc": roc_auc_score(test_y, test_p[m]),
                 "test_pr_auc": average_precision_score(test_y, test_p[m]),
                 "test_brier": brier_score_loss(test_y, test_p[m]),
                 "mean_predicted": float(test_p[m].mean()),
                 "val_mean_roc_auc": float(val_mean.loc[m, "mean_roc_auc"]),
                 "val_mean_pr_auc": float(val_mean.loc[m, "mean_pr_auc"])})
test_scores = pd.DataFrame(rows)
test_scores["roc_gap_test_minus_val"] = test_scores["test_roc_auc"] - test_scores["val_mean_roc_auc"]
base_rate_test = float(test_y.mean())
print(f"Test base rate: {100 * base_rate_test:.2f}% | actual declines: {int(test_y.sum()):,} of {len(test_y):,}")
test_scores.round(4)

Test base rate: 8.11% | actual declines: 4,214 of 51,980


,family,step,test_roc_auc,test_pr_auc,test_brier,mean_predicted,val_mean_roc_auc,val_mean_pr_auc,roc_gap_test_minus_val
0,logreg,baseline,0.8668,0.5069,0.0545,0.0647,0.8364,0.3290,0.0304
1,logreg,step2_rate_level,0.8677,0.5128,0.0537,0.0848,0.8368,0.3288,0.0308
2,logreg,step3_rate_change,0.8648,0.4777,0.0563,0.0675,0.8372,0.3288,0.0276
3,random_forest,baseline,0.8857,0.4665,0.0549,0.0691,0.8680,0.3120,0.0177
4,random_forest,step2_rate_level,0.8779,0.4330,0.0587,0.0965,0.8564,0.3066,0.0216
5,random_forest,step3_rate_change,0.8802,0.4441,0.0558,0.0902,0.8462,0.3142,0.0340
6,hist_gb,baseline,0.8915,0.4759,0.0545,0.0693,0.8666,0.3139,0.0249
7,hist_gb,step2_rate_level,0.8692,0.4176,0.0608,0.0857,0.8525,0.2972,0.0167
8,hist_gb,step3_rate_change,0.8689,0.4234,0.0588,0.0817,0.8511,0.2987,0.0177


In [10]:
TEST_LOCK = Path("../outputs/phase6_test_lock.json")
test_lock = {"purpose": "Test scores of the nine frozen models, written the first time Test was scored.",
             "train_rows": int(len(train)), "test_rows": int(len(test)), "test_declines": int(test_y.sum()),
             "models": {f"{r.family}|{r.step}": {"roc_auc": float(r.test_roc_auc), "pr_auc": float(r.test_pr_auc),
                                                 "brier": float(r.test_brier)} for r in test_scores.itertuples()}}
if TEST_LOCK.exists():
    old = json.loads(TEST_LOCK.read_text(encoding="utf-8"))
    gap = max(abs(old["models"][k][f] - v[f]) for k, v in test_lock["models"].items() for f in v)
    assert gap < 1e-6, f"The Test scores no longer match the locked file. Delete {TEST_LOCK} only if the data changed on purpose."
    print(f"Test scores reproduce the locked file (largest gap {gap:.1e}).")
else:
    TEST_LOCK.write_text(json.dumps(test_lock, indent=2), encoding="utf-8")
    print(f"Test scores locked in {TEST_LOCK}")

Test scores locked in ..\outputs\phase6_test_lock.json


**Reading the Test scores.** On Test (8.11% declines) the nine frozen models reach ROC-AUC 0.8648 to 0.8915 and PR-AUC 0.4176 to 0.5128. All nine score higher than their Phase 5 validation average, by 0.017 to 0.034 ROC-AUC. At baseline, histogram gradient boosting has the highest Test ROC-AUC (0.8915), ahead of the random forest (0.8857) and logistic regression (0.8668). Logistic regression has the highest PR-AUC at every step (0.5069 at baseline). The recommended model stays the random forest on the baseline, because it was chosen on validation and is not changed after Test is seen. Every baseline model predicts a lower average probability (6.5% to 6.9%) than the actual Test decline rate (8.11%).

## 7. Does the rate help on Test? Bootstrap over the 17 Test quarters

The same method as Phase 5, with the Test quarters as the unit: the 17 quarters are resampled with replacement 2,000 times, and every draw scores each model on the quarters drawn. A step "helps" only if its interval excludes zero **and** the change is at least +0.01 ROC-AUC. The same draws give the interval for every model's ROC-AUC and for the differences between families. The first cell proves the weighted AUC equals scikit-learn.

In [11]:
def weighted_auc(gid, yy, w):
    n_g = gid[-1] + 1
    wp = np.bincount(gid, weights=w * yy, minlength=n_g)
    wn = np.bincount(gid, weights=w * (1.0 - yy), minlength=n_g)
    P, N = wp.sum(), wn.sum()
    if P == 0 or N == 0:
        return np.nan
    neg_before = np.cumsum(wn) - wn
    return float((wp * (neg_before + 0.5 * wn)).sum() / (P * N))


qid = {q: i for i, q in enumerate(test_quarters)}
test_qidx = test["quarter"].map(qid).to_numpy()
n_q = len(qid)

prep = {}
for m in MODELS:
    order = np.argsort(test_p[m], kind="mergesort")
    ss = test_p[m][order]
    gid = np.cumsum(np.r_[True, ss[1:] != ss[:-1]]) - 1
    prep[m] = (gid, test_y[order].astype(float), test_qidx[order])

max_gap = max(abs(weighted_auc(g, yy, np.ones(len(yy))) - test_scores.set_index(["family", "step"]).loc[m, "test_roc_auc"])
              for m, (g, yy, _) in prep.items())
print(f"Largest gap between the NumPy AUC and scikit-learn (equal weights): {max_gap:.2e}")
assert max_gap < 1e-9, "The fast AUC does not match scikit-learn."

Largest gap between the NumPy AUC and scikit-learn (equal weights): 1.11e-16


In [12]:
rng = np.random.default_rng(RANDOM_STATE)
auc_draws = {m: [] for m in MODELS}
t0 = time.time()
for it in range(N_BOOT):
    counts = np.bincount(rng.integers(0, n_q, size=n_q), minlength=n_q).astype(float)
    if (counts[test_qidx] * test_y).sum() == 0 or (counts[test_qidx] * (1 - test_y)).sum() == 0:
        continue
    for m in MODELS:
        gid, yy, qs = prep[m]
        auc_draws[m].append(weighted_auc(gid, yy, counts[qs]))
auc_draws = {m: np.array(v) for m, v in auc_draws.items()}
n_used = len(auc_draws[MODELS[0]])
print(f"{n_used} of {N_BOOT} draws used, {time.time() - t0:.0f}s")

ts = test_scores.set_index(["family", "step"])
roc_ci = pd.DataFrame([{"family": m[0], "step": m[1], "test_roc_auc": ts.loc[m, "test_roc_auc"],
                        "ci_lo": float(np.percentile(auc_draws[m], 2.5)),
                        "ci_hi": float(np.percentile(auc_draws[m], 97.5))} for m in MODELS])
roc_ci.round(4)

2000 of 2000 draws used, 25s


,family,step,test_roc_auc,ci_lo,ci_hi
0,logreg,baseline,0.8668,0.8088,0.9065
1,logreg,step2_rate_level,0.8677,0.8074,0.9081
2,logreg,step3_rate_change,0.8648,0.8184,0.9022
3,random_forest,baseline,0.8857,0.8502,0.9139
4,random_forest,step2_rate_level,0.8779,0.8422,0.9069
5,random_forest,step3_rate_change,0.8802,0.8466,0.9074
6,hist_gb,baseline,0.8915,0.8594,0.9175
7,hist_gb,step2_rate_level,0.8692,0.8344,0.9000
8,hist_gb,step3_rate_change,0.8689,0.8330,0.9007


In [13]:
COMPARISONS = [("step2_minus_baseline", "step2_rate_level", "baseline"),
               ("step3_minus_step2", "step3_rate_change", "step2_rate_level"),
               ("step3_minus_baseline", "step3_rate_change", "baseline")]


def verdict(point, lo, hi):
    if lo > 0 and point >= MIN_GAIN:
        return "helps"
    if lo > 0:
        return "positive but below +0.01"
    if hi < 0:
        return "hurts"
    return "no significant effect"


rows = []
for fam in FAMILIES:
    for name, a, b in COMPARISONS:
        arr = auc_draws[(fam, a)] - auc_draws[(fam, b)]
        lo, hi = np.percentile(arr, [2.5, 97.5])
        point = float(ts.loc[(fam, a), "test_roc_auc"] - ts.loc[(fam, b), "test_roc_auc"])
        rows.append({"family": fam, "comparison": name, "point_delta": point, "ci_lo": float(lo),
                     "ci_hi": float(hi), "n_boot_used": len(arr), "verdict": verdict(point, lo, hi)})
test_deltas = pd.DataFrame(rows)
test_deltas.round(4)

,family,comparison,point_delta,ci_lo,ci_hi,n_boot_used,verdict
0,logreg,step2_minus_baseline,0.0009,-0.0021,0.0035,2000,no significant effect
1,logreg,step3_minus_step2,-0.0028,-0.0205,0.0188,2000,no significant effect
2,logreg,step3_minus_baseline,-0.0020,-0.0195,0.0179,2000,no significant effect
3,random_forest,step2_minus_baseline,-0.0077,-0.0118,-0.0044,2000,hurts
4,random_forest,step3_minus_step2,0.0023,-0.0087,0.0143,2000,no significant effect
5,random_forest,step3_minus_baseline,-0.0054,-0.0175,0.0066,2000,no significant effect
6,hist_gb,step2_minus_baseline,-0.0223,-0.0414,-0.0084,2000,hurts
7,hist_gb,step3_minus_step2,-0.0003,-0.0124,0.0123,2000,no significant effect
8,hist_gb,step3_minus_baseline,-0.0226,-0.0361,-0.0093,2000,hurts


In [14]:
robust_rows = []
for name, _, _ in COMPARISONS:
    sub = test_deltas[test_deltas["comparison"] == name]
    n_help, n_hurt = int((sub["verdict"] == "helps").sum()), int((sub["verdict"] == "hurts").sum())
    if n_help >= 2:
        overall = "robust: helps in at least 2 of 3 families"
    elif n_help == 1:
        overall = "not robust: helps in only 1 family"
    elif n_hurt >= 1:
        overall = f"no gain; hurts in {n_hurt} family/families"
    else:
        overall = "no family shows a gain that clears the rule"
    robust_rows.append({"comparison": name, "families_that_help": n_help,
                        "families_that_hurt": n_hurt, "overall": overall})
pd.DataFrame(robust_rows)

,comparison,families_that_help,families_that_hurt,overall
0,step2_minus_baseline,0,2,no gain; hurts in 2 family/families
1,step3_minus_step2,0,0,no family shows a gain that clears the rule
2,step3_minus_baseline,0,1,no gain; hurts in 1 family/families


In [15]:
# Differences between families at each step, from the same draws.
FAMILY_PAIRS = [("random_forest", "logreg"), ("random_forest", "hist_gb"), ("hist_gb", "logreg")]
rows = []
for step in STEP_ORDER:
    for a, b in FAMILY_PAIRS:
        arr = auc_draws[(a, step)] - auc_draws[(b, step)]
        lo, hi = np.percentile(arr, [2.5, 97.5])
        point = float(ts.loc[(a, step), "test_roc_auc"] - ts.loc[(b, step), "test_roc_auc"])
        rows.append({"step": step, "comparison": f"{a} minus {b}", "point_delta": point,
                     "ci_lo": float(lo), "ci_hi": float(hi), "interval_excludes_zero": bool(lo > 0 or hi < 0)})
family_deltas = pd.DataFrame(rows)
family_deltas.round(4)

,step,comparison,point_delta,ci_lo,ci_hi,interval_excludes_zero
0,baseline,random_forest minus logreg,0.0189,0.0008,0.0470,True
1,baseline,random_forest minus hist_gb,-0.0058,-0.0107,-0.0022,True
2,baseline,hist_gb minus logreg,0.0247,0.0050,0.0558,True
3,step2_rate_level,random_forest minus logreg,0.0103,-0.0077,0.0396,False
4,step2_rate_level,random_forest minus hist_gb,0.0087,-0.0053,0.0274,False
5,step2_rate_level,hist_gb minus logreg,0.0016,-0.0238,0.0366,False
6,step3_rate_change,random_forest minus logreg,0.0154,-0.0116,0.0430,False
7,step3_rate_change,random_forest minus hist_gb,0.0114,0.0000,0.0237,True
8,step3_rate_change,hist_gb minus logreg,0.0041,-0.0197,0.0295,False


In [16]:
# PR-AUC interval for the recommended model only (scikit-learn with quarter weights).
def weighted_ap(y, p, w):
    return average_precision_score(y, p, sample_weight=w)

unit_gap = abs(weighted_ap(test_y, test_p[REC], np.ones(len(test_y))) - ts.loc[REC, "test_pr_auc"])
assert unit_gap < 1e-9
rng2 = np.random.default_rng(RANDOM_STATE)
ap_draws = []
t0 = time.time()
for it in range(N_BOOT):
    counts = np.bincount(rng2.integers(0, n_q, size=n_q), minlength=n_q).astype(float)
    w = counts[test_qidx]
    if (w * test_y).sum() == 0:
        continue
    ap_draws.append(weighted_ap(test_y, test_p[REC], w))
ap_lo, ap_hi = np.percentile(ap_draws, [2.5, 97.5])
print(f"{REC[0]} / {REC[1]}: Test PR-AUC {ts.loc[REC, 'test_pr_auc']:.4f} "
      f"(95% interval {ap_lo:.4f} to {ap_hi:.4f}) against a base rate of {100 * base_rate_test:.2f}% "
      f"| ROC-AUC {ts.loc[REC, 'test_roc_auc']:.4f} (95% interval "
      f"{roc_ci.set_index(['family', 'step']).loc[REC, 'ci_lo']:.4f} to {roc_ci.set_index(['family', 'step']).loc[REC, 'ci_hi']:.4f}) "
      f"| {len(ap_draws)} draws, {time.time() - t0:.0f}s")

random_forest / baseline: Test PR-AUC 0.4665 (95% interval 0.3379 to 0.5738) against a base rate of 8.11% | ROC-AUC 0.8857 (95% interval 0.8502 to 0.9139) | 2000 draws, 26s


**Result on Test.** No step passes the rule (interval excludes zero and a gain of at least +0.01) in any family, so the rate adds nothing robust on Test either, including the 2022-23 hiking cycle. Adding the rate level lowers ROC-AUC for the random forest (-0.0077, interval -0.0118 to -0.0044) and histogram gradient boosting (-0.0223, -0.0414 to -0.0084) and leaves logistic regression unchanged (+0.0009, -0.0021 to +0.0035). Adding the rate change and interaction on top changes nothing significant in any family. The intervals are wide because Test has only 17 quarters, so on Test alone even logistic regression cannot rule out a gain of 0.01 at the second step (upper end +0.0188).

**Families.** At baseline, histogram gradient boosting is ahead of the random forest by 0.0058 (interval 0.0022 to 0.0107), and the random forest is ahead of logistic regression by 0.0189 (0.0008 to 0.0470). The random forest remains the recommended model, because it was chosen on validation, where the two were 0.0014 apart. This difference is reported and is not used to change the choice. The PR-AUC of the recommended model is 0.4665, with a 95% interval of 0.3379 to 0.5738, against a base rate of 8.11%.

## 8. Decision rules on Test

The four rules of section 4 applied to the Test predictions. The F1 threshold is the cut-off found on validation, applied unchanged. "Flagged share" is the share of all counties flagged, "precision" is the share of flagged counties that really declined, "recall" is the share of all declines that were flagged, and "lift" is precision divided by the base rate.

In [17]:
rules_test = []
for m in MODELS:
    for s, name in zip(TOP_SHARES, RULE_NAMES[:3]):
        rules_test.append(rule_row("test", m, name, top_share_counts(test_y, test_p[m], test["quarter"].to_numpy(), s)))
    rules_test.append(rule_row("test", m, RULE_NAMES[3], threshold_counts(test_y, test_p[m], f1_threshold[m]["threshold"])))
rules_test = pd.DataFrame(rules_test)
rules_all = pd.concat([rules_val, rules_test], ignore_index=True)

rec_rules = rules_all[(rules_all["family"] == REC[0]) & (rules_all["step"] == REC[1])]
print(f"Recommended model ({REC[0]}, {REC[1]}): validation against Test")
rec_rules.drop(columns=["family", "step"]).round(4).reset_index(drop=True)

Recommended model (random_forest, baseline): validation against Test


,split,rule,tp,fp,fn,tn,precision,recall,flagged_share,lift
0,validation,top 5% each quarter,1650,3983,2602,104499,0.2929,0.3881,0.0500,7.7662
1,validation,top 10% each quarter,2351,8923,1901,99559,0.2085,0.5529,0.1000,5.5289
2,validation,top 20% each quarter,3082,19467,1170,89015,0.1367,0.7248,0.2000,3.6238
3,validation,F1 threshold from validation,2099,3098,2153,105384,0.4039,0.4937,0.0461,10.7083
4,test,top 5% each quarter,1259,1342,2955,46424,0.4840,0.2988,0.0500,5.9707
5,test,top 10% each quarter,2046,3151,2168,44615,0.3937,0.4855,0.1000,4.8562
6,test,top 20% each quarter,2981,7416,1233,40350,0.2867,0.7074,0.2000,3.5367
7,test,F1 threshold from validation,1824,1435,2390,46331,0.5597,0.4328,0.0627,6.9037


In [18]:
cmp_rules = rules_all[(rules_all["step"] == "baseline") & (rules_all["split"] == "test")]
print("Test, baseline step, all three families: precision and recall of each rule")
cmp_rules.pivot_table(index="rule", columns="family", values=["precision", "recall", "flagged_share"])[
    ["precision", "recall", "flagged_share"]].reindex(RULE_NAMES).round(4)

Test, baseline step, all three families: precision and recall of each rule


precision                        recall          \
family                         hist_gb  logreg random_forest hist_gb  logreg   
rule                                                                           
top 5% each quarter             0.4817  0.5021        0.4840  0.2973  0.3099   
top 10% each quarter            0.3970  0.4035        0.3937  0.4896  0.4976   
top 20% each quarter            0.2879  0.2832        0.2867  0.7103  0.6986   
F1 threshold from validation    0.5865  0.5805        0.5597  0.3949  0.4466   

                                           flagged_share                        
family                       random_forest       hist_gb  logreg random_forest  
rule                                                                            
top 5% each quarter                 0.2988        0.0500  0.0500        0.0500  
top 10% each quarter                0.4855        0.1000  0.1000        0.1000  
top 20% each quarter                0.7074        0.2000  0.2000        0.2000  
F1 threshold from validation        0.4328        0.0546  0.0624        0.0627

**Reading the decision rules.** For the recommended model, flagging the riskiest 10% of counties each quarter catches 48.6% of the Test declines, and 39.4% of the flagged counties decline (4.9 times the Test base rate). Precision is higher on Test than on validation because Test has more declines, so lift is the fair comparison: 6.0, 4.9 and 3.5 on Test against 7.8, 5.5 and 3.6 on validation, for the top 5%, 10% and 20%. The F1 threshold of 0.3386 was fixed on validation and flags 6.3% of Test counties against 4.6% of validation counties, because Test has more declines. At the top 5% and 10%, logistic regression has slightly higher precision than the two tree families (0.502 and 0.404 against about 0.48 and 0.39 to 0.40 at baseline), and the three are about equal at 20%. No intervals were computed for these differences.

## 9. Calibration

Calibration asks whether a predicted probability of 20% really means that about 20% of such counties decline. Test is split into probability deciles and compared with the same check on the validation predictions. No recalibration is applied: the check shows whether one is needed.

In [19]:
def calib_table(y, p):
    d = pd.DataFrame({"p": p, "y": y})
    d["decile"] = pd.qcut(d["p"], 10, duplicates="drop")
    t = d.groupby("decile", observed=True).agg(mean_predicted=("p", "mean"), actual_rate=("y", "mean"), n=("y", "size"))
    return t.reset_index(drop=True)


cal_val = calib_table(val_y, val_p[REC]); cal_val.insert(0, "split", "validation")
cal_test = calib_table(test_y, test_p[REC]); cal_test.insert(0, "split", "test")
cal_rec = pd.concat([cal_val, cal_test]).reset_index().rename(columns={"index": "decile"})
cal_rec["decile"] = cal_rec["decile"] + 1
print(f"Recommended model ({REC[0]}, {REC[1]}):")
print(f"  validation: mean predicted {100 * val_p[REC].mean():.2f}% vs actual {100 * val_y.mean():.2f}%, "
      f"Brier {brier_score_loss(val_y, val_p[REC]):.4f}")
print(f"  test:       mean predicted {100 * test_p[REC].mean():.2f}% vs actual {100 * test_y.mean():.2f}%, "
      f"Brier {brier_score_loss(test_y, test_p[REC]):.4f}")
cal_rec.round(4)

Recommended model (random_forest, baseline):
  validation: mean predicted 5.39% vs actual 3.77%, Brier 0.0297
  test:       mean predicted 6.91% vs actual 8.11%, Brier 0.0549


,decile,split,mean_predicted,actual_rate,n
0,1,validation,0.0035,0.0039,11274
1,2,validation,0.0042,0.0032,11273
2,3,validation,0.0047,0.0032,11273
3,4,validation,0.0055,0.0035,11274
4,5,validation,0.0070,0.0076,11273
5,6,validation,0.0101,0.0084,11273
6,7,validation,0.0167,0.0147,11274
7,8,validation,0.0306,0.0227,11273
8,9,validation,0.0759,0.0549,11273
9,10,validation,0.3806,0.2549,11274


In [20]:
rows = []
for m in MODELS:
    ct, cv = calib_table(test_y, test_p[m]), calib_table(val_y, val_p[m])
    rows.append({"family": m[0], "step": m[1],
                 "val_mean_pred_pct": 100 * val_p[m].mean(), "val_actual_pct": 100 * val_y.mean(),
                 "test_mean_pred_pct": 100 * test_p[m].mean(), "test_actual_pct": 100 * test_y.mean(),
                 "test_brier": brier_score_loss(test_y, test_p[m]),
                 "test_deciles_underconfident": int((ct["actual_rate"] > ct["mean_predicted"]).sum()),
                 "test_deciles": len(ct)})
calib_all = pd.DataFrame(rows)
calib_all.round(4)

,family,step,val_mean_pred_pct,val_actual_pct,test_mean_pred_pct,test_actual_pct,test_brier,test_deciles_underconfident,test_deciles
0,logreg,baseline,5.1253,3.7717,6.4652,8.107,0.0545,10,10
1,logreg,step2_rate_level,4.5638,3.7717,8.4788,8.107,0.0537,5,10
2,logreg,step3_rate_change,4.4999,3.7717,6.7461,8.107,0.0563,9,10
3,random_forest,baseline,5.3886,3.7717,6.9058,8.107,0.0549,10,10
4,random_forest,step2_rate_level,5.7027,3.7717,9.6485,8.107,0.0587,7,10
5,random_forest,step3_rate_change,5.9070,3.7717,9.0161,8.107,0.0558,6,10
6,hist_gb,baseline,5.4200,3.7717,6.9315,8.107,0.0545,10,10
7,hist_gb,step2_rate_level,5.7634,3.7717,8.5679,8.107,0.0608,8,10
8,hist_gb,step3_rate_change,5.2501,3.7717,8.1697,8.107,0.0588,8,10


**Reading the calibration.** For the recommended model the average predicted probability is 6.91% on Test against an actual 8.11%, and it is below the actual rate in all 10 deciles (for example 1.3% predicted against 2.9% actual in decile 6). On validation the direction is the opposite: 5.39% predicted against 3.77% actual, with the top decile at 38.1% predicted against 25.5% actual. A possible reason is that the models carry over the decline rate of the years they were trained on: in 9 of the 11 folds the validation year had a lower rate than its training years, and Test has a higher rate than Train (8.11% against 4.85%). This was not tested separately. The consequence is that the ranking is useful, but a predicted probability is not a stable estimate of the share that will decline. A recalibration fitted on validation would move the probabilities in the wrong direction for Test, so none is applied. Adding the rate level raises the average predicted probability (8.48% for logistic regression, 9.65% for the random forest), but it does not raise the ROC-AUC.

## 10. Stability over time: each Test quarter

The Test period has 17 quarters with very different decline rates. For every quarter this shows the base rate, the ROC-AUC, the PR-AUC and the recall at the riskiest 10% of that quarter, for the recommended model, and then the ROC-AUC of all nine models side by side. A quarter without both outcomes has no AUC.

In [21]:
def quarter_metrics(m):
    out = []
    for q in test_quarters:
        mask = (test["quarter"] == q).to_numpy()
        yq, pq = test_y[mask], test_p[m][mask]
        both = 0 < yq.sum() < len(yq)
        k = max(1, int(round(0.10 * len(yq))))
        flag = np.zeros(len(yq), dtype=bool); flag[np.argsort(-pq, kind="mergesort")[:k]] = True
        out.append({"quarter": pd.Timestamp(q).date(), "family": m[0], "step": m[1], "n": int(mask.sum()),
                    "positives": int(yq.sum()), "base_rate_pct": 100 * yq.mean(),
                    "roc_auc": roc_auc_score(yq, pq) if both else np.nan,
                    "pr_auc": average_precision_score(yq, pq) if yq.sum() > 0 else np.nan,
                    "top10_recall": (flag & (yq == 1)).sum() / yq.sum() if yq.sum() > 0 else np.nan,
                    "top10_precision": (flag & (yq == 1)).sum() / flag.sum()})
    return out


per_quarter = pd.DataFrame([r for m in MODELS for r in quarter_metrics(m)])
rec_q = per_quarter[(per_quarter["family"] == REC[0]) & (per_quarter["step"] == REC[1])]
print(f"Recommended model ({REC[0]}, {REC[1]}) in each Test quarter:")
rec_q.drop(columns=["family", "step"]).round(4).reset_index(drop=True)

Recommended model (random_forest, baseline) in each Test quarter:


,quarter,n,positives,base_rate_pct,roc_auc,pr_auc,top10_recall,top10_precision
0,2022-01-01,3020,39,1.2914,0.7555,0.2269,0.4359,0.0563
1,2022-04-01,3019,63,2.0868,0.9626,0.5365,0.8889,0.1854
2,2022-07-01,3021,249,8.2423,0.7897,0.3717,0.4056,0.3344
3,2022-10-01,3060,712,23.2680,0.8793,0.7086,0.3539,0.8235
4,2023-01-01,3060,671,21.9281,0.8416,0.6291,0.3502,0.7680
5,2023-04-01,3059,98,3.2037,0.8497,0.2622,0.5408,0.1732
6,2023-07-01,3060,269,8.7908,0.8838,0.5507,0.5725,0.5033
7,2023-10-01,3063,380,12.4061,0.9261,0.7296,0.5947,0.7386
8,2024-01-01,3063,129,4.2116,0.9015,0.2886,0.6434,0.2712
9,2024-04-01,3063,10,0.3265,0.9606,0.1127,0.8000,0.0261


In [22]:
pq_pivot = per_quarter.pivot_table(index="quarter", columns=["family", "step"], values="roc_auc")
pq_pivot = pq_pivot[MODELS]
pq_pivot.insert(0, "base_rate_pct", rec_q.set_index("quarter")["base_rate_pct"].round(2))
print("ROC-AUC per Test quarter, all nine models")
pq_pivot.round(3)

ROC-AUC per Test quarter, all nine models


family     base_rate_pct   logreg                                     \
step                     baseline step2_rate_level step3_rate_change   
quarter                                                                
2022-01-01          1.29    0.709            0.716             0.734   
2022-04-01          2.09    0.969            0.970             0.973   
2022-07-01          8.24    0.835            0.841             0.851   
2022-10-01         23.27    0.902            0.904             0.893   
2023-01-01         21.93    0.847            0.849             0.865   
2023-04-01          3.20    0.817            0.815             0.793   
2023-07-01          8.79    0.893            0.897             0.913   
2023-10-01         12.41    0.906            0.905             0.896   
2024-01-01          4.21    0.883            0.885             0.892   
2024-04-01          0.33    0.943            0.935             0.921   
2024-07-01          5.28    0.873            0.876             0.886   
2024-10-01          4.62    0.862            0.867             0.861   
2025-01-01          4.07    0.883            0.884             0.884   
2025-04-01         17.06    0.783            0.785             0.786   
2025-07-01         13.39    0.869            0.874             0.878   
2025-10-01          6.42    0.909            0.911             0.912   
2026-01-01          1.07    0.951            0.948             0.945   

family     random_forest                                     hist_gb  \
step            baseline step2_rate_level step3_rate_change baseline   
quarter                                                                
2022-01-01         0.755            0.766             0.797    0.778   
2022-04-01         0.963            0.952             0.950    0.966   
2022-07-01         0.790            0.830             0.735    0.801   
2022-10-01         0.879            0.872             0.880    0.883   
2023-01-01         0.842            0.835             0.843    0.845   
2023-04-01         0.850            0.838             0.845    0.845   
2023-07-01         0.884            0.867             0.866    0.897   
2023-10-01         0.926            0.923             0.926    0.928   
2024-01-01         0.902            0.897             0.895    0.908   
2024-04-01         0.961            0.942             0.938    0.963   
2024-07-01         0.910            0.910             0.899    0.907   
2024-10-01         0.899            0.893             0.891    0.903   
2025-01-01         0.906            0.896             0.898    0.903   
2025-04-01         0.830            0.834             0.819    0.836   
2025-07-01         0.908            0.896             0.890    0.908   
2025-10-01         0.918            0.910             0.913    0.919   
2026-01-01         0.964            0.966             0.967    0.965   

family                                         
step       step2_rate_level step3_rate_change  
quarter                                        
2022-01-01            0.686             0.748  
2022-04-01            0.965             0.968  
2022-07-01            0.830             0.821  
2022-10-01            0.887             0.872  
2023-01-01            0.837             0.838  
2023-04-01            0.839             0.843  
2023-07-01            0.885             0.860  
2023-10-01            0.910             0.923  
2024-01-01            0.895             0.896  
2024-04-01            0.953             0.944  
2024-07-01            0.890             0.898  
2024-10-01            0.891             0.886  
2025-01-01            0.898             0.894  
2025-04-01            0.818             0.819  
2025-07-01            0.902             0.896  
2025-10-01            0.909             0.914  
2026-01-01            0.958             0.963

In [23]:
stab = per_quarter.groupby(["family", "step"])["roc_auc"].agg(["min", "median", "max", "std"])
stab["weakest_quarter"] = per_quarter.loc[per_quarter.groupby(["family", "step"])["roc_auc"].idxmin(), ["family", "step", "quarter"]].set_index(["family", "step"])["quarter"]
stab.reindex(MODELS).round(4)

min  median     max     std  \
family        step                                                
logreg        baseline           0.7092  0.8832  0.9695  0.0635   
              step2_rate_level   0.7158  0.8836  0.9698  0.0615   
              step3_rate_change  0.7336  0.8857  0.9727  0.0593   
random_forest baseline           0.7555  0.9015  0.9637  0.0585   
              step2_rate_level   0.7661  0.8957  0.9657  0.0515   
              step3_rate_change  0.7351  0.8905  0.9674  0.0584   
hist_gb       baseline           0.7782  0.9032  0.9658  0.0545   
              step2_rate_level   0.6855  0.8912  0.9647  0.0658   
              step3_rate_change  0.7478  0.8938  0.9676  0.0565   

                                weakest_quarter  
family        step                               
logreg        baseline               2022-01-01  
              step2_rate_level       2022-01-01  
              step3_rate_change      2022-01-01  
random_forest baseline               2022-01-01  
              step2_rate_level       2022-01-01  
              step3_rate_change      2022-07-01  
hist_gb       baseline               2022-01-01  
              step2_rate_level       2022-01-01  
              step3_rate_change      2022-01-01

## 11. Rates outside the training range on Test

Tree models cannot predict beyond the values they saw in Train. Test contains the 2022-23 hiking cycle, so this shows how much of Test lies outside the range of Train for the two rate features.

In [24]:
rng_rows = []
for feat in ("rate_level", "rate_roc_12m"):
    lo, hi = train[feat].min(), train[feat].max()
    outside = (test[feat] < lo) | (test[feat] > hi)
    rng_rows.append({"feature": feat, "train_min": lo, "train_max": hi, "test_min": test[feat].min(),
                     "test_max": test[feat].max(), "test_pct_outside_train_range": 100 * outside.mean()})
range_check = pd.DataFrame(rng_rows)
range_check.round(3)

,feature,train_min,train_max,test_min,test_max,test_pct_outside_train_range
0,rate_level,2.684,7.160,3.098,7.200,5.893
1,rate_roc_12m,-1.420,1.112,-1.020,3.266,29.357


In [25]:
lo_l, hi_l = train["rate_level"].min(), train["rate_level"].max()
lo_c, hi_c = train["rate_roc_12m"].min(), train["rate_roc_12m"].max()
per_q_range = (test.assign(level_out=(test["rate_level"] < lo_l) | (test["rate_level"] > hi_l),
                           change_out=(test["rate_roc_12m"] < lo_c) | (test["rate_roc_12m"] > hi_c))
               .groupby("quarter")[["level_out", "change_out"]].mean().mul(100).round(1))
per_q_range.index = [pd.Timestamp(q).date() for q in per_q_range.index]
per_q_range.columns = ["rate_level_pct_outside", "rate_change_pct_outside"]
per_q_range

,rate_level_pct_outside,rate_change_pct_outside
2022-01-01,0.0,0.0
2022-04-01,0.0,0.0
2022-07-01,0.0,100.0
2022-10-01,0.0,100.0
2023-01-01,0.0,100.0
2023-04-01,0.0,100.0
2023-07-01,0.0,100.0
2023-10-01,100.0,0.0
2024-01-01,0.0,0.0
2024-04-01,0.0,0.0


**Stability over time.** For the recommended model the quarterly ROC-AUC runs from 0.7555 (2022-Q1, 39 declines) to 0.9637 (2026-Q1, 33 declines), with a median of 0.9015. Twelve of the 17 quarters are at or above 0.85. The five below are in 2022-23 except 2025-Q2 (0.8297). From 2023-Q4 on, every quarter except 2025-Q2 is above 0.89. Quarters with few declines are noisy: 2024-Q2 has only 10. The recall at the riskiest 10% depends on how many counties decline: in 2022-Q4 and 2023-Q1 (22% to 23% declines) a 10% list can catch at most about 43% to 46% of them and catches 35%, with 82% and 77% of the flagged counties declining. The weakest quarter is 2022-Q1 for 8 of the 9 models, the quarter right after the embargo.

**Rates outside the training range.** For the rate level, 5.9% of Test rows are outside the Train range, all in 2023-Q4. For the 12-month rate change, 29.4% are outside it: every county in the five quarters from 2022-Q3 to 2023-Q3, the core of the hiking cycle. In those quarters the effect of adding the rate change on the random forest is mixed: a large drop in 2022-Q3 (0.830 to 0.735) and small changes in the other four. This does not show that the range explains the results.

## 12. Save the results

The tables go into Postgres, where the dashboard reads them. The two lock files written in sections 4 and 6 are the only files that go into git.

In [26]:
rec_scores = ts.loc[REC]
eval_summary = pd.DataFrame([
    ("recommended_family", REC[0]), ("recommended_step", REC[1]),
    ("train_rows", len(train)), ("test_rows", len(test)), ("test_declines", int(test_y.sum())),
    ("test_base_rate", base_rate_test),
    ("test_roc_auc", float(rec_scores["test_roc_auc"])), ("test_pr_auc", float(rec_scores["test_pr_auc"])),
    ("test_brier", float(rec_scores["test_brier"])),
    ("test_pr_auc_ci_lo", float(ap_lo)), ("test_pr_auc_ci_hi", float(ap_hi)),
    ("f1_threshold", f1_threshold[REC]["threshold"]),
], columns=["key", "value"]).astype({"value": str})

test_out = test[["county_fips", "quarter", "label_quarter_end", "decline_label"]].copy()
test_out["quarter"] = test_out["quarter"].dt.date
test_out["label_quarter_end"] = test_out["label_quarter_end"].dt.date
for (family, step), p in test_p.items():
    test_out[f"proba_{family}_{step}"] = p

tables = {
    "phase6_test_scores": test_scores.merge(roc_ci[["family", "step", "ci_lo", "ci_hi"]], on=["family", "step"]).rename(
        columns={"ci_lo": "test_roc_ci_lo", "ci_hi": "test_roc_ci_hi"}),
    "phase6_test_deltas": test_deltas,
    "phase6_family_deltas": family_deltas,
    "phase6_thresholds": thr_table,
    "phase6_decision_rules": rules_all,
    "phase6_calibration": cal_rec,
    "phase6_calibration_all": calib_all,
    "phase6_per_quarter": per_quarter,
    "phase6_range_check": range_check,
    "phase6_eval_summary": eval_summary,
    "phase6_test_predictions": test_out,
}
for name, df in tables.items():
    df.to_sql(name, engine, if_exists="replace", index=False)
with engine.connect() as conn:
    for name in tables:
        print(f"{name:28s} {conn.execute(text(f'SELECT COUNT(*) FROM {name}')).scalar():>8,} rows")

phase6_test_scores                  9 rows
phase6_test_deltas                  9 rows
phase6_family_deltas                9 rows
phase6_thresholds                   9 rows
phase6_decision_rules              72 rows
phase6_calibration                 20 rows
phase6_calibration_all              9 rows
phase6_per_quarter                153 rows
phase6_range_check                  2 rows
phase6_eval_summary                12 rows
phase6_test_predictions        51,980 rows


## Status

Phase 6 is complete and verified on the real data: the thresholds were locked before Test was scored, the nine frozen models were scored on Test once, the Test lock is written, the fast AUC matches scikit-learn (1.1e-16), and the eleven tables are saved. On Test the rate adds nothing robust in any family, and adding the rate level lowers ROC-AUC for both tree families. The recommended model (random forest on the baseline) reaches ROC-AUC 0.8857 (interval 0.8502 to 0.9139) and PR-AUC 0.4665 against a base rate of 8.11%. Its probabilities are too low on Test in every decile, while its ranking holds up: the riskiest 10% of counties each quarter contain 48.6% of the declines.